# Mooncake Transfer Engine 概述

在前面的[《PD 分离技术概述》](./05.02_pd_separation_overview.ipynb)中，我们看到 Prefill 与 Decode 解耦后，Prefill 节点产生的 KV Cache 必须及时、可靠地传输到 Decode 节点。传输数据量大、时延敏感、内存布局复杂等问题，决定了 PD 分离的性能不仅取决于推理引擎本身，也取决于底层数据传输能力。

HIXL 为昇腾 NPU 提供了高带宽的底层传输原语，而在 Mooncake 生态中，Transfer Engine（TE）负责将这些传输能力封装为面向应用的统一接口。下面我们从 TE 的抽象和工作流程出发，进一步说明它如何承接 PD 分离中的 KV Cache 传输需求，以及如何在昇腾环境中使用 Ascend Direct Transport。

Mooncake Transfer Engine（TE）是 Mooncake 的核心传输模块，为跨设备、跨节点的批量数据传输提供统一接口。本节概述 TE 的基本概念、核心特性，以及在昇腾 NPU 上运行传输样例的流程。

本节学习大纲如下：

- Mooncake Transfer Engine 简介
- Ascend Direct Transport
- TE Demo：双 NPU 传输实践


## 1. Mooncake Transfer Engine 简介

Mooncake 的核心是**传输引擎（Transfer Engine，简称 TE）**，它为跨多种存储设备和网络链路的数据批量传输提供了统一接口。TE 支持多种通信协议，包括 TCP、RDMA、CXL/共享内存 以及 基于网络结构的 NVMe（NVMe over Fabric, NVMe-of），旨在为 AI 工作负载提供高速且可靠的数据传输能力。

如下图所示，Mooncake Transfer Engine（TE）作为 Mooncake 的核心模块，对外提供多语言 API，包括 C/C++、Python、Go、Rust 等接口，为上层训练和推理应用（如 vLLM、SGLang）提供高性能的数据传输能力。

<img src="./images/te_architecture.svg" alt="Mooncake Transfer Engine 架构图" width="80%">

上层应用只需通过调用 TE 的统一 API，即可实现跨设备、跨节点的批量数据传输，无需关心底层复杂的网络与存储访问细节。此外，TE 通过 Batch Transfer Interface 封装多种底层传输协议的实现，包括：TCP Transport、RDMA Transport、CXL/SHM Transport、Ascend Direct Transport 等。

### 1.1 核心特性

| 特性 | 说明 |
| --- | --- |
| 多 RDMA 网卡带宽聚合 | 支持多 RDMA 网卡设备，聚合传输带宽 |
| 拓扑感知路径选择 | 根据源/目的内存位置（NUMA 亲和性等）选择最优设备与路径 |
| 网络容错 | 传输失败时自动尝试替代路径完成数据投递 |
| 大规模高性能 | 40GB 数据（LLaMA3-70B 128k token KVCache 规模）下，4×200Gbps 与 8×400Gbps RoCE 网络分别达 87GB/s、190GB/s，约为 TCP 的 2.4x 和 4.6x |
| 异构传输与加速器支持 | 统一支持 TCP、RDMA、EFA、NVMe-oF、NVLink、HIP、CXL 及 Ascend 系列传输；可识别 CUDA/MUSA/HIP/MACA/寒武纪 MLU/Ascend 等加速器内存 |
| LLM 生态广泛采用 | 被 SGLang、vLLM、TensorRT-LLM、vLLM-Ascend、checkpoint-engine、NIXL 等用于传输 KV Cache、Embedding、模型权重等数据 |

### 1.2 核心抽象

TE 围绕两个核心抽象构建：

| 抽象 | 含义 |
| --- | --- |
| **Segment** | 一段可被远端读写的连续地址空间，分为 RAM Segment（DRAM/VRAM）与 NVMeof Segment 两类。每个进程有且仅有一个以 `local_hostname` 命名的 Segment，其他进程通过 `openSegment` 引用并完成读写 |
| **BatchTransfer** | 封装一组读写请求，在 Segment 的若干非连续空间与另一组 Segment 的对应空间间同步数据，支持异步、双向。每条请求指定操作类型（READ/WRITE）、数据长度、本地与远端内存地址，完成状态可通过 `getTransferStatus` 异步轮询 |

### 1.3 典型使用流程

使用 TE 进行数据传输的典型流程为：

```
1. TransferEngine.initialize()    ← 初始化引擎，创建以 local_hostname 命名的 Segment
2. register_memory(ptr, len)      ← 注册本地内存（DRAM/VRAM）
3. openSegment(target_name)       ← 打开远端 Segment 引用
4. batch_transfer_sync_write()    ← 提交批量读写请求
5. unregister_memory(ptr)         ← 注销本地内存
```

> 详细关于 Mooncake TE 的介绍可以访问其 [GitHub](https://github.com/kvcache-ai/Mooncake) 了解。

## 2. Ascend Direct Transport

在昇腾 NPU 上运行 TE 需要依赖于 **Ascend Direct Transport**，该传输层基于华为 CANN 提供的 HIXL 能力构建，是 TE 的原生兼容组件。现在可以直接安装 `mooncake_transfer_engine_npu` 使用，无需手动编译 Mooncake 源码。

用户使用 Ascend Direct Transport 可以实现在 Ascend 环境进行 H2D、D2H、D2D 传输，并且可使用多种通信协议，如 HCCS、RDMA，未来还可实现 Ascend 和 GPU 异构环境的数据传输。

### 2.1 通信协议

| 协议 | 适用场景 |
| --- | --- |
| HCCS | A2 服务器 / A3 超节点机内默认协议，芯片间高速互联 |
| RDMA | 跨机场景；机内可通过 `HCCL_INTRA_ROCE_ENABLE=1` 使用 |

### 2.2 关键注意事项

| 注意点 | 说明 |
| --- | --- |
| 设备初始化 | 调用 `TransferEngine.initialize()` 前必须设置设备，如 `torch.npu.set_device(0)` |
| 内存对齐 | 使用 HCCS 协议时，注册的设备内存地址需 2MB 对齐 |
| 配置文件 | 容器内需确保 `/etc/hccn.conf` 存在（挂载或拷贝） |
| 连接超时 | `ASCEND_CONNECT_TIMEOUT` 控制建链超时（默认 3000ms） |
| 传输超时 | `ASCEND_TRANSFER_TIMEOUT` 控制传输超时（默认 3000ms） |
| 异步传输 | `ASCEND_USE_ASYNC_TRANSFER=1` 启用异步传输模式（需 CANN 8.5+） |
| FabricMem | A3 上 `ASCEND_ENABLE_USE_FABRIC_MEM=1` 启用 FabricMem 模式（需 CANN 9.0+、HDK 25.5+） |

下图展示了 Ascend Direct Transport 的传输能力：NPU 之间通过 HCCS 进行 D2D 直传，Host DRAM 与 NPU 之间通过 RDMA 进行 H2D/D2H 传输，底层均由 CANN ADXL 能力支撑。

<img src="./images/ascend_transport_overview.svg" alt="Ascend Direct Transport 传输示意图" width="75%">

## 3. TE Demo：双 NPU 传输实践

下面提供一个单机双进程的 TE Demo，演示如何在两块 NPU 之间通过 Ascend Direct Transport 进行 D2D 数据传输。

### 3.1 环境准备与依赖检查

执行检查代码单元。该单元会加载 CANN 环境、暴露逻辑设备 0 和 1，并确认 Notebook 环境可以访问 NPU、`torch_npu`、`mooncake.engine` 和 HCCN 配置文件。若 CANN 安装路径不同，请同步修改检查单元和后面的 `run_te_demo.sh`。


In [ ]:
!bash -lc 'python3 -c "import torch, torch_npu; print(torch.npu.device_count())" && python3 -c "from mooncake.engine import TransferEngine; print(\"mooncake.engine import ok\")" '

### 3.2 生成 Demo 程序

执行下面的代码单元生成 `te_demo.py`：


In [ ]:
%%writefile te_demo.py
import os, sys, torch, torch.distributed as dist
import torch_npu
from mooncake.engine import TransferEngine

ALIGN = 2 * 1024 * 1024   # 2 MiB 对齐
NBYTES = 4 * 1024 * 1024
def align_up(p, a): return (p + a - 1) // a * a

def main():
    dev = int(sys.argv[1]); rank = dev % 2
    torch.npu.set_device(dev)
    os.environ.setdefault("MASTER_ADDR", "127.0.0.1")
    os.environ.setdefault("MASTER_PORT", "29500")
    dist.init_process_group("gloo", rank=rank, world_size=2); dist.barrier()

    # 1) 初始化 TE
    te = TransferEngine()
    assert te.initialize("127.0.0.1", "P2PHANDSHAKE", "ascend", "") == 0
    my_sid = f"127.0.0.1:{te.get_rpc_port()}"

    # 2) 在 NPU 上各准备一段对齐内存（src=1, dst=0），并注册可被 TE 访问
    src = torch.ones(ALIGN + NBYTES, dtype=torch.int8, device="npu")
    dst = torch.zeros(ALIGN + NBYTES, dtype=torch.int8, device="npu")
    src_base = align_up(src.data_ptr(), ALIGN)
    dst_base = align_up(dst.data_ptr(), ALIGN)
    assert te.register_memory(src_base, NBYTES) == 0
    assert te.register_memory(dst_base, NBYTES) == 0
    dist.barrier()  # 确保对端也完成注册

    # 3) 交换会话ID与对端接收地址
    gathered = [None, None]
    dist.all_gather_object(gathered, (my_sid, dst_base))
    peer_sid, peer_dst = gathered[1 - rank]

    # 4) 本端把 src 写到对端 dst，然后等待对端也写完
    assert te.batch_transfer_sync_write(peer_sid, [src_base], [peer_dst], [NBYTES]) == 0
    dist.barrier()

    # 5) 校验：本地 dst 被对端写为 1
    off = dst_base - dst.data_ptr()
    ok = (dst.flatten()[off:off + 16].cpu() == 1).all().item()
    if ok:
        print(f"[rank{rank}] te demo success!")
    else:
        print(f"[rank{rank}] te demo failed!")
    dist.destroy_process_group()

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: python3 te_demo.py <dev_id>  # 两个进程分别传 0 和 1")
        sys.exit(1)
    main()

### 3.3 一键启动双进程 Demo

下面的脚本会在当前 Notebook 环境中自动启动两个进程，分别使用逻辑设备 0 和 1。无需再打开两个 shell：


In [ ]:
%%writefile run_te_demo.sh
#!/usr/bin/env bash
set -uo pipefail

ASCEND_ENV_SCRIPT="${ASCEND_ENV_SCRIPT:-/usr/local/Ascend/ascend-toolkit/set_env.sh}"
if [ -f "${ASCEND_ENV_SCRIPT}" ]; then
    # 根据实际 CANN 安装路径设置 ASCEND_ENV_SCRIPT
    source "${ASCEND_ENV_SCRIPT}"
fi

export ASCEND_LOCAL_COMM_RES='{"version" : "1.2"}'
export ASCEND_RT_VISIBLE_DEVICES="${ASCEND_RT_VISIBLE_DEVICES:-0,1}"

python3 te_demo.py 0 &
rank0_pid=$!
python3 te_demo.py 1 &
rank1_pid=$!

rank0_status=0
rank1_status=0
wait "${rank0_pid}" || rank0_status=$?
wait "${rank1_pid}" || rank1_status=$?

if [ "${rank0_status}" -ne 0 ] || [ "${rank1_status}" -ne 0 ]; then
    echo "TE Demo failed: rank0=${rank0_status}, rank1=${rank1_status}"
    exit 1
fi

echo "TE Demo run success!"

执行以下单元，一键拉起两个 TE 进程并查看运行输出：


In [ ]:
!bash run_te_demo.sh

### 3.4 查看运行结果

两个进程正常运行后，Notebook 输出中应看到如下结果：

```
[rank0] te demo success!
[rank1] te demo success!
```

### 3.5 Demo 工作流程解析

下图概括了上述 Demo 的完整工作流程：两个进程各自初始化 TE、注册内存，通过 `all_gather_object` 交换会话 ID 与对端接收地址，随后分别发起批量写传输并校验结果：

<img src="./images/te_demo_flow.svg" alt="TE Demo 工作流程图" width="80%">

结合上面的 Demo，TE 在昇腾 NPU 上的主要工作流程如下：

1. **两个进程分别在各自的 NPU 上创建 `TransferEngine` 实例**。
    - `initialize(local_hostname, metadata_server, protocol, device_name)`，在 NPU 上运行 `protocol` 需填成 `ascend`。
    - TE 会在内部启动一个本地 RPC 服务。

2. **通过 `TransferEngine` 实例调用 `register_memory()` 将内存区域注册到 TE 内部**（实际是告诉 TE 这块内存可以参与 DMA 直拷）。

3. **完成传输对端的信息交换**。
    - Demo 通过 `torch.distributed.all_gather_object()` 交换双方的会话 ID（`my_sid`）和目标地址（`dst_base`）。
    - `P2PHANDSHAKE` 是 TransferEngine 的点对点元数据握手模式，无需单独部署 Metadata Server；多实例场景应使用独立的 Metadata Server。

4. **两个进程分别调用同步批量写接口 `batch_transfer_sync_write(peer_sid, [src_base], [peer_dst], [NBYTES])`**。
    - `[src_base]`、`[peer_dst]`、`[NBYTES]` 分别表示要进行传输的源缓冲区、目标缓冲区、传输字节数的列表，所以是批量传输。
    - 底层调用链为 TransferEngine → Ascend Direct Transport → HIXL/ADXL → HCCS 或 RoCE。


## 课后练习

本节介绍了 Mooncake Transfer Engine 的基本概念、Ascend Direct Transport 与 Demo 运行，请根据学习内容完成以下题目进行自测。

1. （判断题）Mooncake Transfer Engine 的两个核心抽象是 Segment 与 BatchTransfer。

2. （判断题）在昇腾 NPU 上运行 TE，调用 `TransferEngine.initialize()` 前必须先设置设备，如 `torch.npu.set_device(0)`。

3. （单选题）使用 HCCS 协议时，注册的设备内存地址需要对齐到多少？
    A. 4KB
    B. 64KB
    C. 1MB
    D. 2MB

4. （单选题）在 NPU 上运行 TE 时，`initialize()` 的 `protocol` 参数应填为？
    A. `"tcp"`
    B. `"rdma"`
    C. `"ascend"`
    D. `"hccs"`

5. （多选题）Ascend Direct Transport 支持的传输类型有？
    A. H2D（Host → Device）
    B. D2H（Device → Host）
    C. D2D（Device → Device）
    D. NVMe-of 文件读写

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/05.04_answer.txt